In [ ]:
import os
import pandas as pd
import numpy as np
import networkx as nx
import warnings
import time
import json
from pathlib import Path
from gurobipy import Model, GRB, quicksum
import scipy.stats as st
warnings.filterwarnings('ignore')

In [ ]:
BASE = Path(os.environ.get("KEP_DATA_DIR", "../../data"))
POOL_DIR = BASE / 'pool_simulations'
MATRICES_DIR = BASE / 'pool_matrices'
RESULTS_DIR = BASE / 'ABO+DSA' / 'simulation_results'
RESULTS_DIR.mkdir(exist_ok=True)

N_SIMS = 100
LOCI = ['B', 'DR', 'DQ']
ETHCATS = [1, 2, 4, 5, 6, 7]
EQUITY_ETHCATS = [1, 2, 4, 5]     
ETH_LABELS = {1: 'Caucasian', 2: 'Afroamerican', 4: 'Latin', 5: 'Asian',
              6: 'AmInd', 7: 'PacIsl'}

df_pat = pd.read_csv(BASE / 'df_receptores_imputados_final.csv', low_memory=False)
df_pat['WL_ID_CODE'] = df_pat['WL_ID_CODE'].astype('int64')
df_pat['ETHCAT'] = pd.to_numeric(df_pat['ETHCAT'], errors='coerce')

# Parameters
SIM_PARAMS = {
    'TOTAL_TIME':       10 * 12,
    'ARRIVAL_RATE':     1000 / (10 * 12),
    'MEAN_PATIENCE':    65.1552,   
    'MATCH_RUN':        3,
    'WARMUP_MONTHS':    60,   # warmup
    'MAX_CYCLE_LENGTH': 3,
    'SEED_BASE':        42,
    'P':                1100,
    'k_opt': {'antigen': 0, 'allele': 0},
    'Z':     {'antigen': 6, 'allele': 6},
}

In [ ]:
# PER SIM DATA LOADER
def load_sim_data(sim_id):
    pool_df = pd.read_parquet(POOL_DIR / f'pairs_sim_{sim_id:03d}.parquet')
    sim_dir = MATRICES_DIR / f'sim_{sim_id:03d}'
    compat = pd.read_parquet(sim_dir / 'compatibility.parquet').values.astype(np.int8)
    antigen_mm = {L: pd.read_parquet(sim_dir / f'mismatch_antigen_{L}.parquet').values for L in LOCI}
    allele_mm  = {L: pd.read_parquet(sim_dir / f'mismatch_allele_{L}.parquet').values for L in LOCI}
    return {'pool_df': pool_df, 'compat': compat,
            'antigen_mm': antigen_mm, 'allele_mm': allele_mm}

In [ ]:
# BUILD WEIGHT MATRICES 

MAX_ANTIGEN_6LOCI = 6
MAX_ALLELE_6LOCI  = 6

def build_weights_6loci(sim_data):
    am = sim_data['antigen_mm']; al = sim_data['allele_mm']
    sum_antigen = sum(am[L] for L in LOCI).astype(np.int32)
    sum_allele  = sum(al[L] for L in LOCI).astype(np.int32)
    return {
        'antigen': (MAX_ANTIGEN_6LOCI - sum_antigen).astype(np.int32),
        'allele':  (MAX_ALLELE_6LOCI  - sum_allele).astype(np.int32),
        'score_B':  (2 - am['B']).astype(np.int32),
        'score_DR': (2 - am['DR']).astype(np.int32),
        'score_DQ': (2 - am['DQ']).astype(np.int32),
    }

In [ ]:
# GRAPH (ABO+DSA)

def create_graph(waiting_indices, compat):
    G = nx.DiGraph()
    G.add_nodes_from(waiting_indices)
    for i in waiting_indices:
        for j in waiting_indices:
            if i != j and compat[i, j] == 1:
                G.add_edge(j, i)
    return G

def changing_resolution_weights(G, weight_matrix):
    for u, v in G.edges():
        G[u][v]['weight'] = int(weight_matrix[v, u])

In [ ]:
# manuscript Eq. 13

def optimization_with_multipliers(G, pair_ethcat, multipliers,
                                  l=3, k_quality=2, Z=6, P=1100):
    total_cycles = list(nx.simple_cycles(G, length_bound=l))
    valid_cycles = [c for c in total_cycles
                    if all(G[u][v]['weight'] >= k_quality
                           for u, v in zip(c, c[1:] + c[:1]))]

    G_opt = nx.DiGraph()
    if not valid_cycles:
        return G_opt, []

    m = Model("kep_optimization_weighted")
    m.setParam('OutputFlag', 0)
    x = {tuple(c): m.addVar(vtype=GRB.BINARY) for c in valid_cycles}

    def cycle_payoff(c):
        arcs = list(zip(c, c[1:] + c[:1]))
        sum_v  = sum(multipliers.get(int(pair_ethcat[v]), 1.0) for _, v in arcs)
        sum_vw = sum(multipliers.get(int(pair_ethcat[v]), 1.0) * G[u][v]['weight'] / Z
                     for u, v in arcs)
        return sum_v + (1.0 / P) * sum_vw

    m.setObjective(quicksum(x[tuple(c)] * cycle_payoff(c) for c in valid_cycles), GRB.MAXIMIZE)
    for node in G.nodes():
        m.addConstr(quicksum(x[tuple(c)] for c in valid_cycles if node in c) <= 1)
    m.optimize()

    selected = []
    if m.status == GRB.OPTIMAL:
        for c in valid_cycles:
            if x[tuple(c)].X > 0.5:
                selected.append(c)
                for i in range(len(c)):
                    u, v = c[i], c[(i + 1) % len(c)]
                    G_opt.add_edge(u, v, weight=G[u][v]['weight'])
    return G_opt, selected

In [ ]:
# RUN ONE SIMULATION WITH MULTIPLIERS 
def run_simulation_weighted(sim_id, opt_resolution, compat, weights, pair_ethcat,
                            multipliers, params,
                            quality_tracking=False):
    n = compat.shape[0]
    weight_for_obj = weights[opt_resolution]
    k_opt = params['k_opt'][opt_resolution]
    Z     = params['Z'][opt_resolution]
    P_val = params['P']   
    ss = np.random.SeedSequence(params['SEED_BASE'] + sim_id * 1000)
    rng_arr, rng_dep = (np.random.default_rng(s) for s in ss.spawn(2))

    available = set(range(n))
    waiting = []
    arrival_t, departure_t = {}, {}
    historial_cycles = []
    historial_departures = []
    pool_sizes = []
    deadline = {}
    runs_participated = {}
    pool_sizes_by_eth = {e: [] for e in ETHCATS}
    arrivals_by_eth   = {e: 0 for e in ETHCATS}
    departures_by_eth = {e: 0 for e in ETHCATS}

    G_contrib_by_eth = {e: 0.0 for e in ETHCATS}
    G_contrib_total  = 0.0

    if quality_tracking:
        quality = {(res, e): [] for res in ('antigen','allele') for e in ETHCATS}
        quality.update({(loc, e): [] for loc in ('B','DR','DQ') for e in ETHCATS})

    WARMUP = params.get('WARMUP_MONTHS', 0)
    for month in range(params['TOTAL_TIME']):
        counting = month >= WARMUP
        n_arr = rng_arr.poisson(params['ARRIVAL_RATE'])
        if n_arr > 0:
            new_pairs = rng_arr.choice(list(available), size=n_arr, replace=False)
            for p in new_pairs:
                p_int = int(p)
                arrival_t[p_int] = month
                available.discard(p_int)
                waiting.append(p_int)
                deadline[p_int] = month + rng_dep.exponential(params['MEAN_PATIENCE'])
                e = int(pair_ethcat[p_int])
                if counting and e in arrivals_by_eth: arrivals_by_eth[e] += 1

        if (month + 1) % params['MATCH_RUN'] == 0 and len(waiting) >= 2:
            if counting: pool_sizes.append(len(waiting))
            for e_ps in (ETHCATS if counting else []):
                pool_sizes_by_eth[e_ps].append(sum(1 for w in waiting if int(pair_ethcat[w]) == e_ps))
            for _w in waiting:
                runs_participated[_w] = runs_participated.get(_w, 0) + 1
            G = create_graph(waiting, compat)
            changing_resolution_weights(G, weight_for_obj)
            G_opt, selected = optimization_with_multipliers(
                G, pair_ethcat, multipliers,
                l=params['MAX_CYCLE_LENGTH'], k_quality=k_opt, Z=Z, P=params['P'])

            for u, v in (G_opt.edges() if counting else []):
                contrib = 1.0 + weight_for_obj[v, u] / (P_val * Z)
                G_contrib_total += contrib
                e_v = int(pair_ethcat[v])
                if e_v in G_contrib_by_eth:
                    G_contrib_by_eth[e_v] += contrib

            if quality_tracking:
                for u, v in (G_opt.edges() if counting else []):
                    e = int(pair_ethcat[v])
                    if e not in arrivals_by_eth: continue
                    quality[('antigen', e)].append(int(weights['antigen'][v, u]))
                    quality[('allele',  e)].append(int(weights['allele'][v, u]))
                    quality[('B',  e)].append(int(weights['score_B'][v, u]))
                    quality[('DR', e)].append(int(weights['score_DR'][v, u]))
                    quality[('DQ', e)].append(int(weights['score_DQ'][v, u]))

            historial_cycles.extend(selected if counting else [])
            cycled = {p for c in selected for p in c}
            waiting = [w for w in waiting if w not in cycled]
            for p_int in cycled: departure_t[int(p_int)] = month

        departed_now = [w for w in waiting if deadline[w] <= month]
        if departed_now:
            ds = set(departed_now)
            waiting = [w for w in waiting if w not in ds]
            for p_int in (departed_now if counting else []):
                historial_departures.append(p_int)
                e = int(pair_ethcat[p_int])
                if e in departures_by_eth:
                    departures_by_eth[e] += 1

    n_arr_tot = sum(arrivals_by_eth.values())
    n_tx_tot = sum(len(c) for c in historial_cycles)
    F_total = n_tx_tot / max(n_arr_tot, 1)
    L_total = len(historial_departures) / max(n_arr_tot, 1)
    F_per_eth = {e: sum(1 for c in historial_cycles for p in c if int(pair_ethcat[p]) == e)
                       / max(arrivals_by_eth.get(e, 0), 1) for e in ETHCATS}
    L_per_eth = {e: departures_by_eth[e] / max(arrivals_by_eth.get(e, 0), 1) for e in ETHCATS}
    G_per_eth = {e: G_contrib_by_eth[e] / max(arrivals_by_eth.get(e, 0), 1) for e in ETHCATS}
    G_total   = G_contrib_total / max(n_arr_tot, 1)

    out = {
        'sim_id': sim_id, 'opt_resolution': opt_resolution,
        'total_arrivals': n_arr_tot, 'total_transplants': n_tx_tot,
        'total_departures': len(historial_departures),
        'arrivals_by_eth': arrivals_by_eth, 'departures_by_eth': departures_by_eth,
        'F_per_eth': F_per_eth, 'L_per_eth': L_per_eth,
        'F_total': F_total, 'L_total': L_total,
        'G_per_eth': G_per_eth, 'G_total': G_total,
        'historial_cycles': historial_cycles,
        'avg_pool_size': float(np.mean(pool_sizes)) if pool_sizes else 0.0,
        'avg_pool_size_by_eth': {e: float(np.mean(pool_sizes_by_eth[e])) if pool_sizes_by_eth[e] else 0.0 for e in ETHCATS},
    }
    if quality_tracking:
        # waiting times
        wt_by_eth = {e: [] for e in ETHCATS}
        cycled_set = {p for c in historial_cycles for p in c}
        for p in cycled_set:
            if p in runs_participated:
                wt = runs_participated[p]
                e = int(pair_ethcat[p])
                if e in wt_by_eth: wt_by_eth[e].append(wt)
        out['quality'] = quality
        out['waiting_times_by_eth'] = wt_by_eth
    return out

In [ ]:
# PRELOAD: load all 100 sim data + weights + pair_ethcat into memory once
t0 = time.time()
all_sim_data = []
all_weights = []
all_pair_ethcat = []
all_compat = []
for sim_id in range(N_SIMS):
    sd = load_sim_data(sim_id)
    ws = build_weights_6loci(sd)
    eth_join = sd['pool_df'].merge(df_pat[['WL_ID_CODE', 'ETHCAT']], on='WL_ID_CODE', how='left')
    pe = pd.to_numeric(eth_join['ETHCAT'], errors='coerce').fillna(-1).astype(int).values
    all_compat.append(sd['compat'])
    all_weights.append(ws)
    all_pair_ethcat.append(pe)
    all_sim_data.append({'pool_df': sd['pool_df']})
print(f'Done in {time.time()-t0:.1f}s. {len(all_compat)} sims in memory.')

In [ ]:
# OBJECTIVE FUNCTION: run 100 sims, return mean F per eth + F total + mean arrivals per eth

def objective_function(multipliers, opt_resolution):
    F_per_eth_all = {e: [] for e in EQUITY_ETHCATS}
    arr_per_eth_all = {e: [] for e in EQUITY_ETHCATS}
    F_totals = []
    G_per_eth_all = {e: [] for e in EQUITY_ETHCATS}
    G_totals = []
    for sim_id in range(N_SIMS):
        res = run_simulation_weighted(
            sim_id, opt_resolution,
            all_compat[sim_id], all_weights[sim_id], all_pair_ethcat[sim_id],
            multipliers, SIM_PARAMS, quality_tracking=False)
        for e in EQUITY_ETHCATS:
            F_per_eth_all[e].append(res['F_per_eth'][e])
            arr_per_eth_all[e].append(res['arrivals_by_eth'].get(e, 0))
        F_totals.append(res['F_total'])
        for e in EQUITY_ETHCATS:
            G_per_eth_all[e].append(res['G_per_eth'][e])
        G_totals.append(res['G_total'])
    return {
        'F_per_eth':       {e: float(np.mean(F_per_eth_all[e])) for e in EQUITY_ETHCATS},
        'F_total':         float(np.mean(F_totals)),
        'G_per_eth':       {e: float(np.mean(G_per_eth_all[e])) for e in EQUITY_ETHCATS},
        'G_total':         float(np.mean(G_totals)),
        'arrivals_by_eth': {e: float(np.mean(arr_per_eth_all[e])) for e in EQUITY_ETHCATS},
    }



In [ ]:
# RAWLSIAN SEARCH: freeze and continue (maximin incumbent)
# Each iteration boosts the worst active subpopulation by step and evaluates it again over 100 sims.
#   Rule 1: freeze a subpopulation once its value exceeds the population value.
#   Rule 2: freeze the worst active subpopulation if it has not improved for kappa iterations
#           (a structural floor) and continue with the remaining subpopulations.
#   The incumbent is tracked by maximin (maximise the worst subpopulation). The search stops
#   when all subpopulations are frozen or the iteration cap N_max = |S| / step is reached.
#   kappa = m = P (the population size).

def equity_gap(F_per_eth, F_total, arrivals_by_eth):
    """Equity gap: weighted sum of the absolute differences between each F(s) and the population F(P). Reporting metric, not the optimization criterion."""
    total_arr = sum(arrivals_by_eth.values()) or 1.0
    return sum((arrivals_by_eth[e] / total_arr) * abs(F_per_eth[e] - F_total) for e in EQUITY_ETHCATS)

def rawlsian_search(opt_resolution, step=0.00001, kappa=None, max_iters=None,
                    verbose=True, save_history_to=None):
    if kappa is None:
        kappa = SIM_PARAMS['P']                        # m, the population size
    if max_iters is None:
        max_iters = int(len(EQUITY_ETHCATS) / step)    # N_max = |S| / step
    tag = f'opt={opt_resolution}'
    multipliers = {e: 1.0 for e in ETHCATS}
    frozen = {e: False for e in EQUITY_ETHCATS}
    history = []
    best_active, stall = None, 0

    ev = objective_function(multipliers, opt_resolution)
    G_e, G_pob = ev['G_per_eth'], ev['G_total']
    F_e, F_pob = ev['F_per_eth'], ev['F_total']
    for e in EQUITY_ETHCATS:                            # freeze groups already above the population
        if G_e[e] > G_pob:
            frozen[e] = True
    OPT = min(G_e.values())
    best = {'iter': 0, 'multipliers': dict(multipliers), 'F_per_eth': dict(F_e), 'F_total': F_pob,
            'G_per_eth': dict(G_e), 'G_total': G_pob, 'min_G': OPT,
            'equity_gap': equity_gap(F_e, F_pob, ev['arrivals_by_eth'])}
    history.append({'iter': 0, 'multipliers': dict(multipliers), 'G_per_eth': dict(G_e),
                    'G_total': G_pob, 'min_G': OPT, 'frozen': dict(frozen)})
    if verbose:
        print(f'[{tag}] iter 0: min_G={OPT:.5f}  G_pob={G_pob:.5f}  '
              f'frozen={[e for e in EQUITY_ETHCATS if frozen[e]]}')

    for it in range(1, max_iters + 1):
        active = [e for e in EQUITY_ETHCATS if not frozen[e]]
        if not active:
            if verbose:
                print(f'[{tag}] STOP iter {it}: all subpopulations frozen.')
            break
        s_min = min(active, key=lambda e: G_e[e])       # worst active subpopulation
        multipliers[s_min] += step

        ev = objective_function(multipliers, opt_resolution)
        G_e, G_pob = ev['G_per_eth'], ev['G_total']
        F_e, F_pob = ev['F_per_eth'], ev['F_total']
        gap = equity_gap(F_e, F_pob, ev['arrivals_by_eth'])

        for e in EQUITY_ETHCATS:                         # Rule 1: freeze whoever reached the population
            if G_e[e] > G_pob and not frozen[e]:
                frozen[e] = True

        mn = min(G_e.values())                           # maximin incumbent
        if mn > OPT:
            OPT = mn
            best = {'iter': it, 'multipliers': dict(multipliers), 'F_per_eth': dict(F_e), 'F_total': F_pob,
                    'G_per_eth': dict(G_e), 'G_total': G_pob, 'min_G': OPT, 'equity_gap': gap}

        act = [e for e in EQUITY_ETHCATS if not frozen[e]]    # Rule 2: on a stall, freeze and continue
        if act:
            m_active = min(G_e[e] for e in act)
            if best_active is None or m_active > best_active + 1e-9:
                best_active, stall = m_active, 0
            else:
                stall += 1
                if stall >= kappa:
                    stuck = min(act, key=lambda e: G_e[e])
                    frozen[stuck] = True
                    if verbose:
                        print(f'[{tag}] iter {it}: freeze {stuck} (no improvement in {kappa} iters).')
                    best_active, stall = None, 0

        history.append({'iter': it, 'multipliers': dict(multipliers), 'G_per_eth': dict(G_e),
                        'G_total': G_pob, 'min_G': mn, 'equity_gap': gap, 'frozen': dict(frozen)})
        if verbose and it % 10 == 0:
            print(f'[{tag}] iter {it}: min_G={mn:.5f}  OPT={OPT:.5f}  gap={gap:.5f}  active={len(act)}')
        if save_history_to and it % 20 == 0:
            with open(save_history_to, 'w') as f:
                json.dump({'best': best, 'history': history, 'opt_resolution': opt_resolution}, f, indent=2)

    if save_history_to:
        with open(save_history_to, 'w') as f:
            json.dump({'best': best, 'history': history, 'opt_resolution': opt_resolution}, f, indent=2)
    if verbose:
        print(f'[{tag}] BEST: iter={best["iter"]}  min_G={best["min_G"]:.5f}  equity_gap={best["equity_gap"]:.5f}')
    return {'best': best, 'history': history, 'opt_resolution': opt_resolution}


In [ ]:
# RUN — opt = antigen

results_antigen = rawlsian_search(
    'antigen', step=0.00001,
    save_history_to=RESULTS_DIR / 'rawlsian_search_ABO_DSA_6loci_antigen.json'
)

In [ ]:
# RUN — opt = allele

results_allele = rawlsian_search(
    'allele', step=0.00001,
    save_history_to=RESULTS_DIR / 'rawlsian_search_ABO_DSA_6loci_allele.json'
)

In [ ]:
# CONSOLIDATE BEST MULTIPLIERS PER SCENARIO

best_multipliers_per_scenario = {
    'antigen': results_antigen['best']['multipliers'],
    'allele':  results_allele['best']['multipliers'],
}

print('Best multipliers per scenario:')
for opt_res, m in best_multipliers_per_scenario.items():
    line = '  ' + opt_res + ': ' + ', '.join(
        f'{ETH_LABELS[e]}={m[e]:.4f}' for e in EQUITY_ETHCATS)
    print(line)

with open(RESULTS_DIR / 'rawlsian_best_multipliers_ABO_DSA_6loci.json', 'w') as f:
    json.dump(best_multipliers_per_scenario, f, indent=2)
print(f"\nSaved consolidated best multipliers.")

In [ ]:
# APPLY BEST MULTIPLIERS 
RESOLUTIONS = ('antigen', 'allele')

all_results = {res: [] for res in RESOLUTIONS}
t0 = time.time()
for opt_res in RESOLUTIONS:
    m = best_multipliers_per_scenario[opt_res]
    print(f"  opt={opt_res} with multipliers {m}")
    for sim_id in range(N_SIMS):
        res = run_simulation_weighted(
            sim_id, opt_res,
            all_compat[sim_id], all_weights[sim_id], all_pair_ethcat[sim_id],
            m, SIM_PARAMS, quality_tracking=True)
        all_results[opt_res].append(res)



In [ ]:
# RESULT TABLES 

def mean_ci(values, ddof=1, conf=0.95):
    arr = np.asarray([v for v in values if pd.notna(v)], dtype=float)
    if len(arr) < 2:
        if len(arr) == 1: return arr[0], f"{arr[0]:.3f} [-; -]"
        return float('nan'), 'nan'
    m_ = arr.mean(); s = arr.std(ddof=ddof)
    low, high = st.t.interval(conf, len(arr) - 1, loc=m_, scale=s / np.sqrt(len(arr)))
    return m_, f"{m_:.3f} [{low:.3f}; {high:.3f}]"

def build_results_table(rs):
    rows = []
    for e in ETHCATS:
        arr_e = np.mean([r['arrivals_by_eth'][e] for r in rs])
        tx_e = np.mean([r['F_per_eth'][e] * r['arrivals_by_eth'][e] for r in rs])
        F_vals = [r['F_per_eth'][e] for r in rs]
        L_vals = [r['L_per_eth'][e] for r in rs]
        _, txt_F = mean_ci(F_vals); _, txt_L = mean_ci(L_vals)
        ant_vals = [np.mean(r['quality'][('antigen', e)]) for r in rs if r['quality'][('antigen', e)]]
        all_vals = [np.mean(r['quality'][('allele',  e)]) for r in rs if r['quality'][('allele',  e)]]
        _, txt_ant = mean_ci(ant_vals); _, txt_all = mean_ci(all_vals)
        B_v  = [np.mean(r['quality'][('B',  e)]) for r in rs if r['quality'][('B',  e)]]
        dr_v = [np.mean(r['quality'][('DR', e)]) for r in rs if r['quality'][('DR', e)]]
        dq_v = [np.mean(r['quality'][('DQ', e)]) for r in rs if r['quality'][('DQ', e)]]
        _, txt_B = mean_ci(B_v); _, txt_dr = mean_ci(dr_v); _, txt_dq = mean_ci(dq_v)
        wt_flat = [w for r in rs for w in r['waiting_times_by_eth'][e]]
        wt_mean = np.mean(wt_flat) if wt_flat else float('nan')
        still = round(1 - np.mean(F_vals) - np.mean(L_vals), 3)
        rows.append({
            'Ethnicity(s)': e, 'Arrivals': round(arr_e, 2), 'Transplants': round(tx_e, 2),
            'F(s) (Matched)': txt_F, 'HLA(s) Antigen': txt_ant, 'HLA(s) Allele': txt_all,
            'Waiting Time': mean_ci([np.mean(r['waiting_times_by_eth'][e]) for r in rs if r['waiting_times_by_eth'][e]])[1],
            'Pool Size': mean_ci([r['avg_pool_size_by_eth'][e] for r in rs])[1],
            'L(s) (Left Unmatched)': txt_L, '1-F(s)-L(s) (Still in KEP)': still,
            'HLA B': txt_B, 'HLA DR': txt_dr, 'HLA DQ': txt_dq,
        })
    F_tot = [r['F_total'] for r in rs]; L_tot = [r['L_total'] for r in rs]
    _, txt_F_tot = mean_ci(F_tot); _, txt_L_tot = mean_ci(L_tot)
    arr_tot = np.mean([r['total_arrivals'] for r in rs])
    tx_tot  = np.mean([r['total_transplants'] for r in rs])
    ant_ps=[]; all_ps=[]; B_ps=[]; dr_ps=[]; dq_ps=[]; wt_ps=[]
    for r in rs:
        ap = [v for e in ETHCATS for v in r['quality'][('antigen', e)]]
        lp = [v for e in ETHCATS for v in r['quality'][('allele',  e)]]
        bp = [v for e in ETHCATS for v in r['quality'][('B', e)]]
        drp = [v for e in ETHCATS for v in r['quality'][('DR', e)]]
        dqp = [v for e in ETHCATS for v in r['quality'][('DQ', e)]]
        wp = [w for e in ETHCATS for w in r['waiting_times_by_eth'][e]]
        if ap: ant_ps.append(np.mean(ap))
        if lp: all_ps.append(np.mean(lp))
        if bp: B_ps.append(np.mean(bp))
        if drp: dr_ps.append(np.mean(drp))
        if dqp: dq_ps.append(np.mean(dqp))
        if wp: wt_ps.append(np.mean(wp))
    _, txt_ant_t = mean_ci(ant_ps); _, txt_all_t = mean_ci(all_ps)
    _, txt_B_t = mean_ci(B_ps); _, txt_dr_t = mean_ci(dr_ps); _, txt_dq_t = mean_ci(dq_ps)
    still_t = round(1 - np.mean(F_tot) - np.mean(L_tot), 3)
    rows.append({
        'Ethnicity(s)': 'Entire Population', 'Arrivals': round(arr_tot, 2),
        'Transplants': round(tx_tot, 2), 'F(s) (Matched)': txt_F_tot,
        'HLA(s) Antigen': txt_ant_t, 'HLA(s) Allele': txt_all_t,
        'Waiting Time': mean_ci(wt_ps)[1],
        'Pool Size': mean_ci([r['avg_pool_size'] for r in rs])[1],
        'L(s) (Left Unmatched)': txt_L_tot, '1-F(s)-L(s) (Still in KEP)': still_t,
        'HLA B': txt_B_t, 'HLA DR': txt_dr_t, 'HLA DQ': txt_dq_t,
    })
    return pd.DataFrame(rows)

tables = {}
for opt_res in RESOLUTIONS:
    print(f"\n========== RESULTS (Rawlsian weights) — opt={opt_res} ==========\n")
    tbl = build_results_table(all_results[opt_res])
    tables[opt_res] = tbl
    display(tbl)

# Save raw Excel
out_path = RESULTS_DIR / 'results_ABO_DSA_6loci_2scenarios_rawlsian.xlsx'
with pd.ExcelWriter(out_path) as writer:
    for opt_res, tbl in tables.items():
        tbl.to_excel(writer, sheet_name=f'opt_{opt_res}', index=False)
print(f'Saved: {out_path}')

In [ ]:
# RANK TESTS PER ETHNICITY

from scipy.stats import wilcoxon, binomtest

TESTED_ETHCATS = [1, 2, 4, 5, 6, 7]
SMALL_ETHCATS = {6, 7}

def _mean_or_nan(lst):
    return float(np.mean(lst)) if len(lst) else float('nan')
def _F_eth(r, e):
    return r['F_per_eth'][e] if r['arrivals_by_eth'].get(e, 0) > 0 else float('nan')
def _L_eth(r, e):
    return r['L_per_eth'][e] if r['arrivals_by_eth'].get(e, 0) > 0 else float('nan')

METRIC_EXTRACTORS = {
    'F(s) (Matched)':        (_F_eth, lambda r: r['F_total']),
    'L(s) (Left Unmatched)': (_L_eth, lambda r: r['L_total']),
    'HLA(s) Antigen': (lambda r,e: _mean_or_nan(r['quality'][('antigen',e)]),
                       lambda r: _mean_or_nan([v for e2 in ETHCATS for v in r['quality'][('antigen',e2)]])),
    'HLA(s) Allele':  (lambda r,e: _mean_or_nan(r['quality'][('allele',e)]),
                       lambda r: _mean_or_nan([v for e2 in ETHCATS for v in r['quality'][('allele',e2)]])),
    'Waiting Time':   (lambda r,e: _mean_or_nan(r['waiting_times_by_eth'][e]),
                       lambda r: _mean_or_nan([w for e2 in ETHCATS for w in r['waiting_times_by_eth'][e2]])),
    'HLA B':          (lambda r,e: _mean_or_nan(r['quality'][('B',e)]),
                       lambda r: _mean_or_nan([v for e2 in ETHCATS for v in r['quality'][('B',e2)]])),
    'HLA DR':         (lambda r,e: _mean_or_nan(r['quality'][('DR',e)]),
                       lambda r: _mean_or_nan([v for e2 in ETHCATS for v in r['quality'][('DR',e2)]])),
    'HLA DQ':         (lambda r,e: _mean_or_nan(r['quality'][('DQ',e)]),
                       lambda r: _mean_or_nan([v for e2 in ETHCATS for v in r['quality'][('DQ',e2)]])),
}

def rank_test_metric(results_for_scenario, eth_fn, overall_fn, ethcats):
    overall_vals = np.array([overall_fn(r) for r in results_for_scenario], dtype=float)
    out = {}
    for e in ethcats:
        eth_vals = np.array([eth_fn(r, e) for r in results_for_scenario], dtype=float)
        mask = ~(np.isnan(eth_vals) | np.isnan(overall_vals))
        diffs = eth_vals[mask] - overall_vals[mask]
        non_zero = diffs[diffs != 0]
        if len(non_zero) > 0:
            try:    _, p_w = wilcoxon(non_zero, alternative='two-sided')
            except ValueError: p_w = float('nan')
        else: p_w = float('nan')
        n_pos = int((diffs > 0).sum()); n_neg = int((diffs < 0).sum())
        n_tot = n_pos + n_neg
        p_s = binomtest(n_pos, n_tot, 0.5, alternative='two-sided').pvalue if n_tot > 0 else float('nan')
        out[e] = {'p_wilcoxon': p_w, 'p_sign': p_s, 'n_pos': n_pos, 'n_neg': n_neg,
                  'n_used': int(mask.sum()),
                  'median_diff': float(np.median(diffs)) if len(diffs) else float('nan')}
    return out

rank_results_all = {}
for opt_res in RESOLUTIONS:
    rank_results_all[opt_res] = {}
    for metric_name, (eth_fn, overall_fn) in METRIC_EXTRACTORS.items():
        rank_results_all[opt_res][metric_name] = rank_test_metric(
            all_results[opt_res], eth_fn, overall_fn, TESTED_ETHCATS)

def _flags(r):
    f = ''
    if not np.isnan(r['p_wilcoxon']) and r['p_wilcoxon'] < 0.05: f += 'W'
    if not np.isnan(r['p_sign'])     and r['p_sign']     < 0.05: f += 'S'
    return f

for opt_res in RESOLUTIONS:
    print(f"\n===== opt={opt_res} (Rawlsian) — flags  [W = Wilcoxon p<.05, S = sign test p<.05] =====")
    header = f"{'Metric':24s}" + "".join(
        f"{(ETH_LABELS[e] + ('*' if e in SMALL_ETHCATS else '')):>14s}" for e in TESTED_ETHCATS)
    print(header); print('-' * len(header))
    for m in METRIC_EXTRACTORS:
        row = f"{m:24s}"
        for e in TESTED_ETHCATS:
            row += f"{('[' + _flags(rank_results_all[opt_res][m][e]) + ']'):>14s}"
        print(row)

def _paired_rank_tests(diffs):
    non_zero = diffs[diffs != 0]
    if len(non_zero) > 0:
        try:    _, p_w = wilcoxon(non_zero, alternative='two-sided')
        except ValueError: p_w = float('nan')
    else:
        p_w = float('nan')
    n_pos = int((diffs > 0).sum())
    n_neg = int((diffs < 0).sum())
    n_tot = n_pos + n_neg
    p_s = binomtest(n_pos, n_tot, 0.5, alternative='two-sided').pvalue if n_tot > 0 else float('nan')
    return {
        'p_wilcoxon':  p_w,
        'p_sign':      p_s,
        'n_pos':       n_pos,
        'n_neg':       n_neg,
        'n_used':      int(len(diffs)),
        'median_diff': float(np.median(diffs)) if len(diffs) else float('nan'),
    }

def rank_test_metric_pair(results_A, results_B, eth_fn, ethcats):
    n = min(len(results_A), len(results_B))
    out = {}
    for e in ethcats:
        vals_A = np.array([eth_fn(results_A[i], e) for i in range(n)], dtype=float)
        vals_B = np.array([eth_fn(results_B[i], e) for i in range(n)], dtype=float)
        mask = ~(np.isnan(vals_A) | np.isnan(vals_B))
        diffs = vals_A[mask] - vals_B[mask]
        out[e] = _paired_rank_tests(diffs)
    return out

# Pairwise between resolutions
rank_results_pairwise = {res_A: {} for res_A in RESOLUTIONS}
for res_A in RESOLUTIONS:
    for res_B in RESOLUTIONS:
        if res_A == res_B:
            continue
        rank_results_pairwise[res_A][res_B] = {}
        for metric_name, (eth_fn, _) in METRIC_EXTRACTORS.items():
            rank_results_pairwise[res_A][res_B][metric_name] = rank_test_metric_pair(
                all_results[res_A], all_results[res_B], eth_fn, TESTED_ETHCATS)

print("\n\n===== PAIRWISE BETWEEN-RESOLUTION comparisons (paired by sim_id) =====")
print("    [W = Wilcoxon p<.05, S = sign test p<.05]")
seen = set()
for res_A in RESOLUTIONS:
    for res_B in RESOLUTIONS:
        if res_A == res_B: continue
        key = tuple(sorted([res_A, res_B]))
        if key in seen: continue
        seen.add(key)
        a, b = key
        print(f"\n--- {a} vs {b} ---")
        _hdr = f"{'Metric':24s}" + "".join(f"{ETH_LABELS[e]:>14s}" for e in TESTED_ETHCATS)
        print(_hdr); print("-" * len(_hdr))
        for m in METRIC_EXTRACTORS:
            _row = f"{m:24s}"
            for e in TESTED_ETHCATS:
                _row += f"{('[' + _flags(rank_results_pairwise[a][b][m][e]) + ']'):>14s}"
            print(_row)



In [ ]:

from openpyxl import Workbook
from openpyxl.styles import Font

ALPHA = 0.05
METRIC_COLUMNS = list(METRIC_EXTRACTORS.keys())

OTHER_RESS = {opt_res: [r for r in RESOLUTIONS if r != opt_res] for opt_res in RESOLUTIONS}

rank_results_pairwise_overall = {res_A: {} for res_A in RESOLUTIONS}
for res_A in RESOLUTIONS:
    for res_B in RESOLUTIONS:
        if res_A == res_B: continue
        rank_results_pairwise_overall[res_A][res_B] = {}
        for metric_name, (_, overall_fn) in METRIC_EXTRACTORS.items():
            A = all_results[res_A]; B = all_results[res_B]
            n = min(len(A), len(B))
            vals_A = np.array([overall_fn(A[i]) for i in range(n)], dtype=float)
            vals_B = np.array([overall_fn(B[i]) for i in range(n)], dtype=float)
            mask = ~(np.isnan(vals_A) | np.isnan(vals_B))
            rank_results_pairwise_overall[res_A][res_B][metric_name] = _paired_rank_tests(vals_A[mask] - vals_B[mask])

def _pairwise_marks_overall(opt_res, metric):
    others = OTHER_RESS[opt_res]
    marks = ''
    for sym, other in zip(['†', '‡'], others):
        r = rank_results_pairwise_overall[opt_res][other][metric]
        if (not np.isnan(r['p_wilcoxon'])) and r['p_wilcoxon'] < ALPHA:
            marks += sym
    return marks

def _pairwise_marks(opt_res, metric, eth_code):
    if eth_code not in TESTED_ETHCATS: return ''
    others = OTHER_RESS[opt_res]
    marks = ''
    for sym, other in zip(['†', '‡'], others):
        if other not in rank_results_pairwise.get(opt_res, {}):
            continue
        r = rank_results_pairwise[opt_res][other][metric][eth_code]
        if (not np.isnan(r['p_wilcoxon'])) and r['p_wilcoxon'] < ALPHA:
            marks += sym
    return marks

wb = Workbook(); wb.remove(wb.active)
for opt_res in RESOLUTIONS:
    ws = wb.create_sheet(f'opt_{opt_res}')
    tbl = tables[opt_res]
    for col_idx, col_name in enumerate(tbl.columns, start=1):
        c = ws.cell(row=1, column=col_idx, value=col_name); c.font = Font(bold=True)
    for row_pos, (_, row) in enumerate(tbl.iterrows(), start=2):
        eth_raw = row['Ethnicity(s)']
        is_eth = isinstance(eth_raw, (int, np.integer))
        eth_code = int(eth_raw) if is_eth else None
        for col_idx, col_name in enumerate(tbl.columns, start=1):
            val = row[col_name]
            if col_name == 'Ethnicity(s)' and is_eth:
                val = ETH_LABELS.get(eth_code, str(eth_raw)) + ('*' if eth_code in SMALL_ETHCATS else '')
            cell = ws.cell(row=row_pos, column=col_idx, value=val)
            if col_name in METRIC_COLUMNS and eth_code in TESTED_ETHCATS:
                r = rank_results_all[opt_res][col_name][eth_code]
                bold      = (not np.isnan(r['p_wilcoxon'])) and r['p_wilcoxon'] < ALPHA
                underline = (not np.isnan(r['p_sign']))     and r['p_sign']     < ALPHA
                if bold or underline:
                    cell.font = Font(bold=bold, underline='single' if underline else None)
                # Pairwise vs other resolutions: append †/‡
                marks = _pairwise_marks(opt_res, col_name, eth_code)
                if marks:
                    if isinstance(val, str):
                        cell.value = val + marks
                    elif isinstance(val, (int, float)) and not (isinstance(val, float) and np.isnan(val)):
                        cell.number_format = f'0.000"{marks}"' 
            if col_name in METRIC_COLUMNS and (eth_code is None):
                marks = _pairwise_marks_overall(opt_res, col_name)
                if marks:
                    if isinstance(val, str):
                        cell.value = val + marks
                    elif isinstance(val, (int, float)) and not (isinstance(val, float) and np.isnan(val)):
                        cell.number_format = f'0.000"{marks}"'
    for col_idx, col_name in enumerate(tbl.columns, start=1):
        col_letter = chr(64 + col_idx) if col_idx <= 26 else 'A' + chr(64 + col_idx - 26)
        ws.column_dimensions[col_letter].width = max(14, len(col_name) + 2)
    foot = len(tbl) + 4
    # Note the best multipliers used
    m = best_multipliers_per_scenario[opt_res]
    ws.cell(row=foot, column=1,
            value='Rawlsian equity weights used: ' + ', '.join(
                f'{ETH_LABELS[e]}={m[e]:.4f}' for e in EQUITY_ETHCATS))
    fb = ws.cell(row=foot+2, column=1, value='   bold      = Wilcoxon signed-rank p < 0.05')
    fu = ws.cell(row=foot+3, column=1, value='   underlined = sign test p < 0.05')
    others = OTHER_RESS[opt_res]
    if len(others) >= 1:
        ws.cell(row=foot+4, column=1, value=f'   †  = Wilcoxon signed-rank p < 0.05 — this resolution ({opt_res}) vs {others[0]} (paired by sim_id)')
    if len(others) >= 2:
        ws.cell(row=foot+5, column=1, value=f'   ‡  = Wilcoxon signed-rank p < 0.05 — this resolution ({opt_res}) vs {others[1]} (paired by sim_id)')
    fb.font = Font(bold=True); fu.font = Font(underline='single')

out_path = RESULTS_DIR / 'results_ABO_DSA_6loci_2scenarios_rawlsian_significance.xlsx'
wb.save(out_path)
print(f'Saved: {out_path}')
